# 09 · Interpretabilidad del modelo final y análisis de errores

Este notebook interpreta el **XGBoost ya congelado** y analiza sus errores sobre noviembre-diciembre de 2022. No entrena, no recalibra, no modifica umbrales y no vuelve a seleccionar modelos.

La interpretación combina importancia por ganancia con valores SHAP nativos de XGBoost sobre una muestra estratificada. Los resultados describen asociaciones predictivas del modelo; **no demuestran relaciones causales**.

## Protocolo cerrado

- Modelo: `xgboost_refined`, seleccionado antes de abrir el test.
- Entrenamiento: 2019, 2021 y enero-septiembre de 2022.
- Validación: octubre de 2022.
- Test confirmatorio ya ejecutado: noviembre-diciembre de 2022.
- Este notebook reutiliza el artefacto congelado y los CSV finales del 08, pero no ejecuta el 08 ni llama a `fit`.
- Los datos de 2023 no se leen; permanecen limitados al análisis descriptivo de viajes.

## Entorno, rutas y controles de integridad

In [ ]:
# %pip install pandas numpy scikit-learn xgboost matplotlib

from pathlib import Path
import gc
import json
import pickle
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
try:
    import xgboost
except Exception as error:
    raise RuntimeError(
        'XGBoost no puede cargarse. Usa el mismo kernel que ejecutó los notebooks 07 y 08.'
    ) from error
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
MODEL_DATA_DIR = PROJECT_ROOT / 'notebooks' / 'Datos modelado'
FEATURES_DIR = MODEL_DATA_DIR / 'estacion_hora_features'
ARTIFACTS_DIR = MODEL_DATA_DIR / 'validacion_externa'
TEST_RESULTS_DIR = MODEL_DATA_DIR / 'test_final_2022'
OUTPUT_DIR = MODEL_DATA_DIR / 'interpretabilidad_modelo_final'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'risk_class_1h'
TIME_COLUMN = 'fecha_hora_local'
TEST_PERIODS = ('202211', '202212')
CLASS_NAMES = {0: 'estable', 1: 'riesgo_vaciado', 2: 'riesgo_saturacion'}
CATEGORICAL_FEATURES = ['station_id', 'tipo_dia']
RANDOM_STATE = 42
CHUNK_SIZE = 100_000
SHAP_ROWS_PER_CLASS = 4_000
MIN_STATION_ROWS = 500
MIN_CLASS_SUPPORT = 20
CRITICAL_HOURS = [8, 18, 19]

CURRENT_VERSIONS = {
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'xgboost': xgboost.__version__,
}
EXPECTED_MODEL_VERSIONS = {'scikit-learn': '1.9.0', 'xgboost': '3.2.0'}
version_mismatches = {
    package: {'expected': expected, 'current': CURRENT_VERSIONS[package]}
    for package, expected in EXPECTED_MODEL_VERSIONS.items()
    if CURRENT_VERSIONS[package] != expected
}
if version_mismatches:
    raise RuntimeError(f'Kernel incompatible con los artefactos congelados: {version_mismatches}')

print('Resultados de entrada:', TEST_RESULTS_DIR)
print('Resultados de interpretabilidad:', OUTPUT_DIR)
print('Versiones:', CURRENT_VERSIONS)

## Carga del artefacto congelado y de las predicciones finales

Se exige la marca de finalización y la auditoría del notebook 08. Solo se abre el XGBoost final; la regresión logística no participa en esta fase.

In [ ]:
completion_sentinel = TEST_RESULTS_DIR / '_EVALUACION_FINAL_COMPLETADA.txt'
audit_08_path = TEST_RESULTS_DIR / 'auditoria_test_final_2022.csv'
predictions_path = TEST_RESULTS_DIR / 'predicciones_test_final_2022.csv'
artifact_path = ARTIFACTS_DIR / 'xgboost_refined_validacion_externa.pkl'

for required_path in [completion_sentinel, audit_08_path, predictions_path, artifact_path]:
    if not required_path.exists():
        raise FileNotFoundError(f'Falta un resultado obligatorio: {required_path}')

audit_08 = pd.read_csv(audit_08_path, encoding='utf-8-sig')
audit_08_values = dict(zip(audit_08['control'], audit_08['value'].astype(str)))
assert audit_08_values['periodos_test_leidos'] == '202211,202212'
assert audit_08_values['dataset_split_exigido'] == 'test'
assert audit_08_values['llamadas_fit_en_notebook'] == '0'
assert audit_08_values['preprocesadores_reajustados'].lower() == 'false'
assert audit_08_values['hiperparametros_modificados'].lower() == 'false'
assert audit_08_values['modelo_final_preestablecido'] == 'xgboost_refined'
assert audit_08_values['datos_2023_leidos'].lower() == 'false'

with artifact_path.open('rb') as file:
    artifact = pickle.load(file)
required_keys = {'model', 'preprocessor', 'features', 'classes', 'frozen_configuration'}
assert required_keys.issubset(artifact)
assert artifact['classes'] == CLASS_NAMES
assert int(artifact['frozen_configuration']['n_estimators']) == 381
assert int(artifact['model'].get_params()['n_estimators']) == 381
MODEL_FEATURES = list(artifact['features'])
xgb_model = artifact['model']
preprocessor = artifact['preprocessor']

predictions_all = pd.read_csv(predictions_path, encoding='utf-8-sig')
predictions = predictions_all.loc[predictions_all['model'].eq('xgboost_refined')].copy()
predictions[TIME_COLUMN] = pd.to_datetime(predictions[TIME_COLUMN], errors='raise')
predictions[TARGET] = predictions[TARGET].astype('int8')
predictions['prediction'] = predictions['prediction'].astype('int8')
assert len(predictions) == int(audit_08_values['filas_evaluadas']) == 379_104
assert not predictions.duplicated([TIME_COLUMN, 'station_id']).any()
assert predictions[TIME_COLUMN].dt.to_period('M').astype(str).isin(['2022-11', '2022-12']).all()
assert set(predictions[TARGET].unique()).issubset(CLASS_NAMES)

print(f'Predicciones XGBoost cargadas: {len(predictions):,}')
print('Modelo congelado:', type(xgb_model).__name__, '· árboles:', xgb_model.get_params()['n_estimators'])
del predictions_all
gc.collect()

## Contexto operativo del test

Se recuperan disponibilidad, capacidad, ubicación, meteorología y flujos recientes para describir errores y preparar el notebook 10. Las etiquetas reales se mantienen únicamente en las tablas de evaluación; la entrada operativa del 10 se exportará sin ellas.

In [ ]:
def period_from_file(file_path: Path) -> str:
    return file_path.stem.rsplit('_', maxsplit=1)[-1]

test_files = [FEATURES_DIR / f'estacion_hora_features_{period}.csv' for period in TEST_PERIODS]
assert all(path.exists() for path in test_files)
assert tuple(period_from_file(path) for path in test_files) == TEST_PERIODS

OPERATIONAL_COLUMNS = [
    TIME_COLUMN, 'station_id', 'station_number', 'station_name', 'address',
    'latitude', 'longitude', 'capacity', 'bikes_available', 'docks_available',
    'reservations_count', 'occupancy_ratio', 'tipo_dia',
    'temperature_median_c', 'relative_humidity_median_pct',
    'wind_speed_median_m_s', 'precipitation_mean_l_m2',
    'departures_count_lag_1h', 'arrivals_count_lag_1h', 'net_flow_lag_1h',
]
CONTEXT_READ_COLUMNS = list(dict.fromkeys(OPERATIONAL_COLUMNS + [TARGET, 'dataset_split']))
context_parts = []
for file_path in test_files:
    for chunk in pd.read_csv(
        file_path, usecols=CONTEXT_READ_COLUMNS, chunksize=CHUNK_SIZE, low_memory=False,
    ):
        eligible = chunk.loc[
            chunk['dataset_split'].eq('test') & chunk[TARGET].notna()
        ].copy()
        if not eligible.empty:
            context_parts.append(eligible)

context = pd.concat(context_parts, ignore_index=True)
context[TIME_COLUMN] = pd.to_datetime(context[TIME_COLUMN], errors='raise')
context[TARGET] = context[TARGET].astype('int8')
assert len(context) == len(predictions)
assert not context.duplicated([TIME_COLUMN, 'station_id']).any()

context = context.drop(columns='dataset_split').rename(columns={TARGET: 'target_context'})
analysis_frame = predictions.merge(
    context, on=[TIME_COLUMN, 'station_id'], how='left', validate='one_to_one',
)
assert analysis_frame['target_context'].notna().all()
assert analysis_frame[TARGET].eq(analysis_frame['target_context']).all()
analysis_frame = analysis_frame.drop(columns='target_context')
analysis_frame['month'] = analysis_frame[TIME_COLUMN].dt.to_period('M').astype(str)
analysis_frame['hour'] = analysis_frame[TIME_COLUMN].dt.hour
analysis_frame['is_false_negative_empty'] = (
    analysis_frame[TARGET].eq(1) & ~analysis_frame['prediction'].eq(1)
)
analysis_frame['is_false_negative_full'] = (
    analysis_frame[TARGET].eq(2) & ~analysis_frame['prediction'].eq(2)
)
analysis_frame['is_critical_false_negative'] = (
    analysis_frame['is_false_negative_empty'] | analysis_frame['is_false_negative_full']
)

print('Filas enriquecidas con contexto operativo:', f'{len(analysis_frame):,}')
display(analysis_frame[[TIME_COLUMN, 'station_id', 'station_name', TARGET, 'prediction']].head())
del context_parts, context
gc.collect()

## Muestra estratificada para TreeSHAP

Se seleccionan hasta 4.000 observaciones por clase real. El sobremuestreo analítico de las clases minoritarias evita que la explicación quede dominada por la clase estable; no altera el modelo ni sus predicciones.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sample_indices = []
for class_value in CLASS_NAMES:
    candidates = predictions.index[predictions[TARGET].eq(class_value)].to_numpy()
    sample_size = min(SHAP_ROWS_PER_CLASS, len(candidates))
    sample_indices.extend(rng.choice(candidates, size=sample_size, replace=False).tolist())
sample_indices = np.array(sample_indices, dtype=int)
rng.shuffle(sample_indices)
sample_metadata = predictions.loc[sample_indices, [
    TIME_COLUMN, 'station_id', TARGET, 'prediction', 'is_error',
    'probability_0', 'probability_1', 'probability_2',
]].copy()
sample_metadata.insert(0, 'prediction_row_index', sample_indices)
sample_key_index = pd.MultiIndex.from_frame(sample_metadata[[TIME_COLUMN, 'station_id']])

SHAP_READ_COLUMNS = list(dict.fromkeys(MODEL_FEATURES + [TIME_COLUMN, 'station_id', TARGET, 'dataset_split']))
shap_raw_parts = []
for file_path in test_files:
    for chunk in pd.read_csv(
        file_path, usecols=SHAP_READ_COLUMNS, chunksize=CHUNK_SIZE, low_memory=False,
    ):
        chunk = chunk.loc[chunk['dataset_split'].eq('test') & chunk[TARGET].notna()].copy()
        if chunk.empty:
            continue
        chunk[TIME_COLUMN] = pd.to_datetime(chunk[TIME_COLUMN], errors='raise')
        chunk_keys = pd.MultiIndex.from_frame(chunk[[TIME_COLUMN, 'station_id']])
        selected = chunk.loc[chunk_keys.isin(sample_key_index)].copy()
        if not selected.empty:
            shap_raw_parts.append(selected.drop(columns=[TARGET, 'dataset_split']))

shap_raw = pd.concat(shap_raw_parts, ignore_index=True)
assert len(shap_raw) == len(sample_metadata)
assert not shap_raw.duplicated([TIME_COLUMN, 'station_id']).any()
shap_sample = sample_metadata.merge(
    shap_raw, on=[TIME_COLUMN, 'station_id'], how='left', validate='one_to_one',
)
assert shap_sample[MODEL_FEATURES].notna().any(axis=1).all()
sample_class_counts = shap_sample[TARGET].value_counts().sort_index()
display(sample_class_counts.rename(index=CLASS_NAMES).rename('shap_rows').to_frame())

X_shap_raw = shap_sample[MODEL_FEATURES]
X_shap = preprocessor.transform(X_shap_raw).astype(np.float32)
transformed_feature_names = np.asarray(preprocessor.get_feature_names_out(), dtype=object)
booster = xgb_model.get_booster()
assert X_shap.shape[1] == len(transformed_feature_names) == booster.num_features()
print('Matriz SHAP:', X_shap.shape)

del shap_raw_parts, shap_raw
gc.collect()

## Importancia por ganancia

La ganancia resume cuánto reducen la función de pérdida las divisiones que usan cada variable transformada. Se exportan ganancia media, ganancia total y número de divisiones. Las variables one-hot se agregan después a su variable original.

In [ ]:
def original_feature_name(transformed_name: str) -> str:
    name = transformed_name.split('__', maxsplit=1)[-1]
    if name.startswith('missingindicator_'):
        name = name.removeprefix('missingindicator_')
    for categorical_feature in CATEGORICAL_FEATURES:
        if name == categorical_feature or name.startswith(f'{categorical_feature}_'):
            return categorical_feature
    return name

gain_score = booster.get_score(importance_type='gain')
total_gain_score = booster.get_score(importance_type='total_gain')
weight_score = booster.get_score(importance_type='weight')
gain_rows = []
for index, transformed_name in enumerate(transformed_feature_names):
    # Según cómo se haya creado la DMatrix, XGBoost puede guardar f0, f1... o nombres explícitos.
    key = str(transformed_name) if str(transformed_name) in total_gain_score else f'f{index}'
    gain_rows.append({
        'transformed_feature': transformed_name,
        'original_feature': original_feature_name(str(transformed_name)),
        'gain': float(gain_score.get(key, 0.0)),
        'total_gain': float(total_gain_score.get(key, 0.0)),
        'split_count': float(weight_score.get(key, 0.0)),
    })
gain_transformed = pd.DataFrame(gain_rows)
gain_original = (
    gain_transformed.groupby('original_feature', as_index=False)
    .agg(total_gain=('total_gain', 'sum'), split_count=('split_count', 'sum'))
)
gain_original['mean_gain_per_split'] = (
    gain_original['total_gain'] / gain_original['split_count'].replace(0, np.nan)
)
gain_original['total_gain_share'] = gain_original['total_gain'] / gain_original['total_gain'].sum()
gain_original = gain_original.sort_values('total_gain', ascending=False).reset_index(drop=True)
gain_original['gain_rank'] = np.arange(1, len(gain_original) + 1)
display(gain_original.head(20))

## TreeSHAP global y por clase

`pred_contribs=True` calcula valores TreeSHAP exactos para el margen de cada clase. La importancia se mide mediante el valor absoluto medio. Las contribuciones de indicadores y categorías one-hot se suman por observación antes de volver a la variable original. La tabla global ofrece dos lecturas: macro, con el mismo peso para cada clase, y ponderada por la prevalencia real del test.

In [ ]:
shap_with_bias = booster.predict(
    xgboost.DMatrix(X_shap), pred_contribs=True, strict_shape=True,
)
n_rows = len(shap_sample)
n_classes = len(CLASS_NAMES)
n_features = len(transformed_feature_names)
if shap_with_bias.ndim == 3:
    assert shap_with_bias.shape == (n_rows, n_classes, n_features + 1)
    shap_values = shap_with_bias[:, :, :-1]
    shap_bias = shap_with_bias[:, :, -1]
elif shap_with_bias.ndim == 2:
    assert shap_with_bias.shape[1] == n_classes * (n_features + 1)
    reshaped = shap_with_bias.reshape(n_rows, n_classes, n_features + 1)
    shap_values = reshaped[:, :, :-1]
    shap_bias = reshaped[:, :, -1]
else:
    raise ValueError(f'Forma SHAP inesperada: {shap_with_bias.shape}')

feature_groups = {}
for index, transformed_name in enumerate(transformed_feature_names):
    original_name = original_feature_name(str(transformed_name))
    feature_groups.setdefault(original_name, []).append(index)

shap_transformed_parts = []
shap_original_parts = []
grouped_contributions = {}
for class_value, class_name in CLASS_NAMES.items():
    transformed_part = pd.DataFrame({
        'class_value': class_value,
        'class_name': class_name,
        'transformed_feature': transformed_feature_names,
        'original_feature': [original_feature_name(str(name)) for name in transformed_feature_names],
        'mean_abs_shap': np.abs(shap_values[:, class_value, :]).mean(axis=0),
        'mean_signed_shap': shap_values[:, class_value, :].mean(axis=0),
    })
    shap_transformed_parts.append(transformed_part)

    for original_name, indices in feature_groups.items():
        contribution = shap_values[:, class_value, indices].sum(axis=1)
        grouped_contributions[(class_value, original_name)] = contribution
        shap_original_parts.append({
            'class_value': class_value,
            'class_name': class_name,
            'original_feature': original_name,
            'mean_abs_shap': float(np.abs(contribution).mean()),
            'mean_signed_shap': float(contribution.mean()),
            'median_shap': float(np.median(contribution)),
            'q25_shap': float(np.quantile(contribution, 0.25)),
            'q75_shap': float(np.quantile(contribution, 0.75)),
        })

shap_transformed = pd.concat(shap_transformed_parts, ignore_index=True)
shap_original = pd.DataFrame(shap_original_parts)
shap_original['shap_rank_within_class'] = (
    shap_original.groupby('class_value')['mean_abs_shap']
    .rank(method='first', ascending=False).astype(int)
)
test_class_share = predictions[TARGET].value_counts(normalize=True).to_dict()
shap_original['test_class_share'] = shap_original['class_value'].map(test_class_share)
shap_original['weighted_abs_shap_component'] = (
    shap_original['mean_abs_shap'] * shap_original['test_class_share']
)
shap_global = (
    shap_original.groupby('original_feature', as_index=False)
    .agg(
        mean_abs_shap_macro=('mean_abs_shap', 'mean'),
        mean_abs_shap_test_weighted=('weighted_abs_shap_component', 'sum'),
    )
    .sort_values('mean_abs_shap_macro', ascending=False)
    .reset_index(drop=True)
)
shap_global['shap_macro_rank'] = np.arange(1, len(shap_global) + 1)
shap_global['shap_test_weighted_rank'] = (
    shap_global['mean_abs_shap_test_weighted'].rank(method='first', ascending=False).astype(int)
)
display(shap_global.head(20))
for class_value in [1, 2]:
    print(CLASS_NAMES[class_value])
    display(
        shap_original.loc[shap_original['class_value'].eq(class_value)]
        .nsmallest(15, 'shap_rank_within_class')
    )

## Dirección asociativa de las variables numéricas

Para facilitar la lectura se calcula la correlación de rangos entre el valor observado y su contribución SHAP. Una correlación positiva indica que valores mayores suelen empujar el margen hacia esa clase; una negativa indica lo contrario. Es una descripción del comportamiento del modelo, no causalidad.

In [ ]:
numeric_features = [feature for feature in MODEL_FEATURES if feature not in CATEGORICAL_FEATURES]
direction_rows = []
for class_value, class_name in CLASS_NAMES.items():
    for feature in numeric_features:
        raw_values = pd.to_numeric(shap_sample[feature], errors='coerce')
        contribution = pd.Series(grouped_contributions[(class_value, feature)], index=shap_sample.index)
        valid = raw_values.notna() & contribution.notna()
        if valid.sum() < 100 or raw_values.loc[valid].nunique() < 3:
            rank_correlation = np.nan
        else:
            rank_correlation = raw_values.loc[valid].rank().corr(contribution.loc[valid].rank())
        if pd.isna(rank_correlation) or abs(rank_correlation) < 0.15:
            direction = 'sin_direccion_monotona_clara'
        elif rank_correlation > 0:
            direction = 'valores_altos_aumentan_contribucion'
        else:
            direction = 'valores_altos_reducen_contribucion'
        direction_rows.append({
            'class_value': class_value, 'class_name': class_name,
            'original_feature': feature, 'rank_correlation_value_shap': rank_correlation,
            'direction_association': direction, 'valid_rows': int(valid.sum()),
        })
direction_table = pd.DataFrame(direction_rows)
shap_factors = shap_original.merge(
    direction_table, on=['class_value', 'class_name', 'original_feature'], how='left',
)
display(
    shap_factors.loc[shap_factors['class_value'].isin([1, 2])]
    .sort_values(['class_value', 'shap_rank_within_class']).groupby('class_value').head(15)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 6))
global_top = shap_global.head(15).sort_values('mean_abs_shap_macro')
axes[0].barh(global_top['original_feature'], global_top['mean_abs_shap_macro'], color='#0B6E99')
axes[0].set_title('Importancia SHAP global macro')
axes[0].set_xlabel('|SHAP| medio')
for axis, class_value, color in [(axes[1], 1, '#D97904'), (axes[2], 2, '#A23B72')]:
    top = (
        shap_original.loc[shap_original['class_value'].eq(class_value)]
        .nsmallest(15, 'shap_rank_within_class').sort_values('mean_abs_shap')
    )
    axis.barh(top['original_feature'], top['mean_abs_shap'], color=color)
    axis.set_title(CLASS_NAMES[class_value].replace('_', ' ').title())
    axis.set_xlabel('|SHAP| medio')
fig.suptitle('TreeSHAP del XGBoost congelado · muestra estratificada')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_global_y_por_clase.png', dpi=160, bbox_inches='tight')
plt.show()

comparison = shap_global.merge(
    gain_original[['original_feature', 'total_gain_share', 'gain_rank']],
    on='original_feature', how='left',
)
comparison['shap_macro_share'] = (
    comparison['mean_abs_shap_macro'] / comparison['mean_abs_shap_macro'].sum()
)
display(comparison.head(20))

## Falsos negativos por mes, hora y estación

El análisis prioriza riesgos reales clasificados como estables u otra clase. Cada tasa conserva su denominador para evitar conclusiones basadas en muy pocos casos.

In [ ]:
def grouped_error_summary(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    work = frame.copy()
    work['is_error_int'] = work['is_error'].astype(int)
    work['real_empty'] = work[TARGET].eq(1).astype(int)
    work['real_full'] = work[TARGET].eq(2).astype(int)
    work['fn_empty'] = work['is_false_negative_empty'].astype(int)
    work['fn_full'] = work['is_false_negative_full'].astype(int)
    work['fp_empty'] = ((~work[TARGET].eq(1)) & work['prediction'].eq(1)).astype(int)
    work['fp_full'] = ((~work[TARGET].eq(2)) & work['prediction'].eq(2)).astype(int)
    summary = (
        work.groupby(group_columns, dropna=False, as_index=False)
        .agg(
            rows=(TARGET, 'size'), errors=('is_error_int', 'sum'),
            support_empty=('real_empty', 'sum'), false_negatives_empty=('fn_empty', 'sum'),
            support_full=('real_full', 'sum'), false_negatives_full=('fn_full', 'sum'),
            false_positives_empty=('fp_empty', 'sum'), false_positives_full=('fp_full', 'sum'),
        )
    )
    summary['error_rate'] = summary['errors'] / summary['rows']
    summary['recall_empty'] = 1 - (
        summary['false_negatives_empty'] / summary['support_empty'].replace(0, np.nan)
    )
    summary['recall_full'] = 1 - (
        summary['false_negatives_full'] / summary['support_full'].replace(0, np.nan)
    )
    summary['critical_support'] = summary['support_empty'] + summary['support_full']
    summary['critical_false_negatives'] = (
        summary['false_negatives_empty'] + summary['false_negatives_full']
    )
    summary['critical_false_negative_rate'] = (
        summary['critical_false_negatives'] / summary['critical_support'].replace(0, np.nan)
    )
    return summary

errors_by_month = grouped_error_summary(analysis_frame, ['month'])
errors_by_hour = grouped_error_summary(analysis_frame, ['hour'])
errors_by_station = grouped_error_summary(analysis_frame, ['station_id'])
station_catalog = (
    analysis_frame.sort_values(TIME_COLUMN)
    .drop_duplicates('station_id', keep='last')[
        ['station_id', 'station_number', 'station_name', 'address', 'latitude', 'longitude', 'capacity']
    ]
)
errors_by_station = errors_by_station.merge(
    station_catalog, on='station_id', how='left', validate='one_to_one'
)

false_negatives = analysis_frame.loc[analysis_frame['is_critical_false_negative']].copy()
false_negatives['real_risk_name'] = false_negatives[TARGET].map(CLASS_NAMES)
false_negatives['predicted_name'] = false_negatives['prediction'].map(CLASS_NAMES)
print(f'Falsos negativos críticos: {len(false_negatives):,}')
display(errors_by_month)

## Horas críticas y estaciones con soporte suficiente

Las 08:00, 18:00 y 19:00 se revisan explícitamente. Una estación entra en la tabla prioritaria si tiene al menos 500 observaciones y soporte mínimo de 20 casos en alguna clase crítica.

In [ ]:
critical_hours_table = errors_by_hour.loc[errors_by_hour['hour'].isin(CRITICAL_HOURS)].copy()
critical_hours_table['priority_reason'] = 'hora_predefinida_por_tasa_de_error'
stations_with_support = errors_by_station.loc[
    errors_by_station['rows'].ge(MIN_STATION_ROWS)
    & (
        errors_by_station['support_empty'].ge(MIN_CLASS_SUPPORT)
        | errors_by_station['support_full'].ge(MIN_CLASS_SUPPORT)
    )
].copy()
stations_with_support['enough_support_empty'] = stations_with_support['support_empty'].ge(MIN_CLASS_SUPPORT)
stations_with_support['enough_support_full'] = stations_with_support['support_full'].ge(MIN_CLASS_SUPPORT)
stations_with_support = stations_with_support.sort_values(
    ['critical_false_negatives', 'errors', 'error_rate'], ascending=False
)

print('Horas críticas:')
display(critical_hours_table)
print('Estaciones prioritarias con soporte suficiente:')
display(stations_with_support.head(25))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(errors_by_hour['hour'], errors_by_hour['error_rate'], marker='o', color='#0B6E99')
for hour in CRITICAL_HOURS:
    axes[0].axvline(hour, color='#D97904', alpha=0.45, linestyle='--')
axes[0].set_title('Tasa de error por hora')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('Tasa de error')
axes[0].grid(alpha=0.25)
top_stations = stations_with_support.head(15).sort_values('critical_false_negatives')
axes[1].barh(top_stations['station_name'], top_stations['critical_false_negatives'], color='#A23B72')
axes[1].set_title('Falsos negativos críticos · estaciones con soporte')
axes[1].set_xlabel('Número de falsos negativos')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'errores_horas_y_estaciones.png', dpi=160, bbox_inches='tight')
plt.show()

## Traducción a factores operativos

Las variables se agrupan por su posible uso operativo. La tabla resultante indica qué señales utiliza el modelo para cada riesgo y cómo se asocian con su contribución, sin afirmar que intervenir sobre una variable cause el resultado.

In [ ]:
def factor_family(feature: str) -> str:
    if feature == 'station_id':
        return 'identidad_y_heterogeneidad_espacial'
    if feature in {'hour', 'day_of_week', 'month', 'week_of_year', 'tipo_dia'}:
        return 'temporal_y_calendario'
    if feature in {'capacity', 'bikes_available', 'docks_available', 'reservations_count', 'occupancy_ratio'}:
        return 'estado_actual_estacion'
    if any(token in feature for token in ['lag_', 'previous_', 'net_flow', 'departures_count', 'arrivals_count']):
        return 'dinamica_historica_reciente'
    if any(token in feature for token in [
        'temperature', 'humidity', 'precipitation', 'wind_', 'radiation', 'pressure', 'weather',
    ]):
        return 'meteorologia'
    return 'otra_senal_contextual'

operational_factors = shap_factors.loc[
    shap_factors['class_value'].isin([1, 2]) & shap_factors['shap_rank_within_class'].le(20)
].copy()
operational_factors['factor_family'] = operational_factors['original_feature'].map(factor_family)
operational_factors['interpretation_scope'] = 'asociacion_predictiva_no_causal'
operational_factors = operational_factors.sort_values(
    ['class_value', 'shap_rank_within_class']
)
display(operational_factors)

## Entradas para el notebook 10

La entrada de predicciones contiene solo información disponible en el instante de predicción, probabilidades y señal de acción del modelo. No incluye la etiqueta real ni indicadores de acierto para impedir que el notebook de recomendaciones use información futura.

In [ ]:
action_signal = {
    0: 'monitorizar',
    1: 'candidata_reposicion_bicicletas',
    2: 'candidata_retirada_bicicletas',
}
input_10_columns = [
    TIME_COLUMN, 'station_id', 'station_number', 'station_name', 'address',
    'latitude', 'longitude', 'capacity', 'bikes_available', 'docks_available',
    'reservations_count', 'occupancy_ratio', 'tipo_dia',
    'temperature_median_c', 'relative_humidity_median_pct',
    'wind_speed_median_m_s', 'precipitation_mean_l_m2',
    'departures_count_lag_1h', 'arrivals_count_lag_1h', 'net_flow_lag_1h',
    'prediction', 'confidence', 'probability_0', 'probability_1', 'probability_2',
]
input_10_predictions = analysis_frame[input_10_columns].copy()
input_10_predictions['predicted_risk_name'] = input_10_predictions['prediction'].map(CLASS_NAMES)
input_10_predictions['model_action_signal'] = input_10_predictions['prediction'].map(action_signal)
input_10_predictions['critical_risk_score'] = input_10_predictions[[
    'probability_1', 'probability_2'
]].max(axis=1)
input_10_predictions['critical_risk_type'] = np.where(
    input_10_predictions['probability_1'].ge(input_10_predictions['probability_2']),
    'riesgo_vaciado', 'riesgo_saturacion',
)
assert TARGET not in input_10_predictions.columns
assert 'is_error' not in input_10_predictions.columns

input_10_factors = operational_factors[[
    'class_value', 'class_name', 'original_feature', 'shap_rank_within_class',
    'mean_abs_shap', 'mean_signed_shap', 'rank_correlation_value_shap',
    'direction_association', 'factor_family', 'interpretation_scope',
]].copy()
input_10_stations = stations_with_support.copy()
input_10_hours = errors_by_hour.sort_values('critical_false_negative_rate', ascending=False).copy()

print('Entrada 10 · predicciones y contexto:', f'{len(input_10_predictions):,}')
print('Entrada 10 · factores:', len(input_10_factors))
print('Entrada 10 · estaciones:', len(input_10_stations))
print('Entrada 10 · horas:', len(input_10_hours))

## Exportación y auditoría final

In [ ]:
exports = {
    'importancia_gain_transformada.csv': gain_transformed,
    'importancia_gain_agregada.csv': gain_original,
    'shap_importancia_transformada_por_clase.csv': shap_transformed,
    'shap_importancia_agregada_por_clase.csv': shap_original,
    'shap_importancia_global.csv': shap_global,
    'comparacion_gain_vs_shap.csv': comparison,
    'shap_direccion_variables_numericas.csv': direction_table,
    'factores_operativos_modelo_final.csv': operational_factors,
    'errores_por_mes_modelo_final.csv': errors_by_month,
    'errores_por_hora_modelo_final.csv': errors_by_hour,
    'errores_horas_08_18_19.csv': critical_hours_table,
    'errores_por_estacion_modelo_final.csv': errors_by_station,
    'estaciones_con_soporte_para_revision.csv': stations_with_support,
    'falsos_negativos_criticos_detalle.csv': false_negatives,
    'muestra_estratificada_shap.csv': sample_metadata,
    'entrada_10_predicciones_y_contexto.csv': input_10_predictions,
    'entrada_10_factores_por_clase.csv': input_10_factors,
    'entrada_10_estaciones_sensibles.csv': input_10_stations,
    'entrada_10_horas_sensibles.csv': input_10_hours,
    'catalogo_estaciones.csv': station_catalog,
}
for file_name, table in exports.items():
    table.to_csv(OUTPUT_DIR / file_name, index=False, encoding='utf-8-sig')

audit_table = pd.DataFrame([
    {'control': 'modelo_interpretado', 'value': 'xgboost_refined'},
    {'control': 'artefacto_congelado', 'value': str(artifact_path)},
    {'control': 'n_estimators', 'value': int(artifact['frozen_configuration']['n_estimators'])},
    {'control': 'llamadas_fit', 'value': 0},
    {'control': 'preprocesador_reajustado', 'value': False},
    {'control': 'hiperparametros_modificados', 'value': False},
    {'control': 'umbrales_modificados', 'value': False},
    {'control': 'periodos_interpretados', 'value': ','.join(TEST_PERIODS)},
    {'control': 'filas_predicciones', 'value': len(predictions)},
    {'control': 'filas_muestra_shap', 'value': len(shap_sample)},
    {'control': 'shap_rows_class_0', 'value': int(sample_class_counts.get(0, 0))},
    {'control': 'shap_rows_class_1', 'value': int(sample_class_counts.get(1, 0))},
    {'control': 'shap_rows_class_2', 'value': int(sample_class_counts.get(2, 0))},
    {'control': 'horas_revision_especial', 'value': ','.join(map(str, CRITICAL_HOURS))},
    {'control': 'min_filas_estacion', 'value': MIN_STATION_ROWS},
    {'control': 'min_soporte_clase_estacion', 'value': MIN_CLASS_SUPPORT},
    {'control': 'datos_2023_leidos', 'value': False},
    {'control': 'interpretacion_causal', 'value': False},
])
audit_table.to_csv(OUTPUT_DIR / 'auditoria_interpretabilidad.csv', index=False, encoding='utf-8-sig')

manifest = {
    'notebook': '09_interpretabilidad_modelo_final_y_analisis_errores.ipynb',
    'model': 'xgboost_refined',
    'fit_calls': 0,
    'test_periods': list(TEST_PERIODS),
    'shap_method': 'xgboost_native_tree_shap_pred_contribs',
    'shap_sample_rows': int(len(shap_sample)),
    'outputs': sorted(list(exports) + [
        'auditoria_interpretabilidad.csv', 'shap_global_y_por_clase.png',
        'errores_horas_y_estaciones.png',
    ]),
}
with (OUTPUT_DIR / 'manifiesto_interpretabilidad.json').open('w', encoding='utf-8') as file:
    json.dump(manifest, file, indent=2, ensure_ascii=False)

assert int(audit_table.loc[audit_table['control'].eq('llamadas_fit'), 'value'].iloc[0]) == 0
assert not bool(audit_table.loc[audit_table['control'].eq('interpretacion_causal'), 'value'].iloc[0])
display(audit_table)
print('Interpretabilidad completada. Resultados guardados en:', OUTPUT_DIR)

## Cómo interpretar esta fase y continuar

1. La importancia identifica señales utilizadas por el modelo, no causas del vaciado o saturación.
2. Compara ganancia y SHAP: coincidencias refuerzan la estabilidad de la lectura; discrepancias deben documentarse.
3. Para estaciones y horas usa siempre soporte, falsos negativos y tasa, no rankings sin denominador.
4. Las probabilidades del test no deben emplearse para recalibrar ni elegir nuevos umbrales.
5. El notebook 10 debe partir de `entrada_10_predicciones_y_contexto.csv` y complementar la señal del modelo con capacidad, disponibilidad, proximidad y restricciones operativas.
6. Las señales de acción exportadas son candidatas técnicas; el notebook 10 será responsable de convertirlas en recomendaciones priorizadas y explicables.